# 🎓 Notebook 4: MLOps Deployment — MLflow & Vertex AI

## AuraCart Retail Analytics — Production Deployment Pipeline

This notebook documents the production deployment workflow for AuraCart's customer segment prediction model:

1. **MLflow Experiment Tracking** — Review and select the champion model
2. **Model Packaging** — Create a unified Scikit-learn Pipeline (preprocessor + classifier)
3. **Artifact Serialization** — Save as `model.joblib` with `requirements.txt`
4. **Google Cloud Storage** — Upload artifacts to GCS bucket
5. **Vertex AI Deployment** — Deploy as a live RESTful prediction endpoint
6. **Endpoint Testing** — Send a prediction request and verify the response

### Syllabus Alignment
- Module 6: Production ML Systems (MLflow, model serialization, Vertex AI, containers)

## Step 1: Setup & Load Model Artifacts

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

import mlflow
import mlflow.sklearn

ARTIFACTS_DIR = os.path.join('..', 'artifacts')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 2: Review MLflow Experiments & Select Champion Model

We query the MLflow tracking server to compare all experiment runs and select the best-performing customer segment classification model based on F1-score.

In [2]:
# Connect to MLflow
mlflow.set_tracking_uri('mlruns')

# List all experiments
experiments = mlflow.search_experiments()
print('=== MLflow Experiments ===')
for exp in experiments:
    print(f'  [{exp.experiment_id}] {exp.name}')

=== MLflow Experiments ===


  [173217744263218082] AuraCart_Customer_Clustering
  [373904052182076952] AuraCart_Segment_Classification
  [641163872348690363] AuraCart_Delivery_Classification
  [650346320980896139] AuraCart_Price_Regression
  [0] Default


In [3]:
# Query Customer Segment classification runs
experiment = mlflow.get_experiment_by_name('AuraCart_Segment_Classification')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                               order_by=['metrics.f1_macro DESC'])
    print('=== Customer Segment Classification Runs ===')
    if len(runs) > 0:
        display_cols = ['run_id', 'params.solver', 'params.C', 'params.class_weight',
                        'metrics.accuracy', 'metrics.f1_macro']
        available_cols = [c for c in display_cols if c in runs.columns]
        print(runs[available_cols].to_string())
        
        # Best run
        best_run = runs.iloc[0]
        print(f'\n\u2705 Best Run ID: {best_run["run_id"]}')
        print(f'   F1 Macro: {best_run.get("metrics.f1_macro", "N/A")}')
    else:
        print('No runs found. Please run Notebook 2 first.')
else:
    print('Experiment not found. Please run Notebook 2 first.')

=== Customer Segment Classification Runs ===
                              run_id params.solver params.C params.class_weight  metrics.accuracy  metrics.f1_macro
0   b4b28eeb3d8d46d4b9047d675d5b9d86         lbfgs      0.1            balanced             0.259          0.236511
1   61c12b1688a045d3bcd9c905b00de15c          saga      1.0            balanced             0.259          0.236511
2   4d777f98a56543cca506b0f3376439c4         lbfgs      0.1            balanced             0.259          0.236511
3   8091ebc5158244c395f1210374611022          saga      1.0            balanced             0.259          0.236511
4   867310d3d94a43c3b4238298b55896e0         lbfgs      0.1            balanced             0.259          0.236511
5   21574a150e684ebc8ddccf7dd4fef12e          saga      1.0            balanced             0.259          0.236511
6   2ffd8de5e3ba4a7a9c900144fa7a122a         lbfgs     10.0            balanced             0.258          0.235776
7   862841023f504f7594be493

## Step 3: Create Unified Prediction Pipeline

We combine the preprocessing pipeline (from Notebook 1) and the trained classification model (from Notebook 2) into a **single Scikit-learn Pipeline object**. This ensures:
- All preprocessing steps are applied consistently during inference
- The deployment artifact is self-contained and portable
- Raw input data can be fed directly without manual transformation

### Why Pipelines?
A Pipeline packages multiple processing steps into a single estimator. This is critical for production because:
1. It eliminates the risk of applying different preprocessing at training vs inference time
2. The Vertex AI pre-built container can load and serve a single Pipeline object
3. It makes the deployment reproducible and version-controlled

In [4]:
# Load the preprocessor and best classification model
preprocessor = joblib.load(os.path.join(ARTIFACTS_DIR, 'preprocessor.pkl'))
best_classifier = joblib.load(os.path.join(ARTIFACTS_DIR, 'best_segment_classifier.pkl'))
le_segment = joblib.load(os.path.join(ARTIFACTS_DIR, 'le_segment.pkl'))
split_data = joblib.load(os.path.join(ARTIFACTS_DIR, 'train_test_split.pkl'))

print('Preprocessor:', preprocessor)
print('\nClassifier:', best_classifier)
print('\nSegment classes:', le_segment.classes_)

Preprocessor: ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['quantity', 'order_month',
                                  'order_day_of_week', 'order_hour',
                                  'shipping_delay_days']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['category', 'payment_method', 'device_type',
                                  'channel'])])

Classifier: LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42,
                   solver='saga')

Segment classes: ['New' 'Returning' 'VIP']


In [5]:
# Create the unified prediction pipeline
# This combines preprocessing + classification into a single object
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', best_classifier)
])

print('\u2705 Unified Pipeline Created:')
print(final_pipeline)

✅ Unified Pipeline Created:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['quantity', 'order_month',
                                                   'order_day_of_week',
                                                   'order_hour',
                                                   'shipping_delay_days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['category', 'payment_method',
                                                   'device_type',
                                                   'channel'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                    

In [6]:
# Verify the pipeline works end-to-end with raw data
X_test_raw = split_data['X_test']
y_segment_test = split_data['y_segment_test']

# The pipeline should accept raw (unprocessed) features
y_pred = final_pipeline.predict(X_test_raw)

# Decode predictions
y_pred_labels = le_segment.inverse_transform(y_pred)

from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(split_data['y_segment_test_enc'], y_pred)

print(f'\u2705 Pipeline verification successful!')
print(f'Accuracy on test set: {accuracy:.4f}')
print(f'\nSample predictions (first 10):')
for i in range(min(10, len(y_pred_labels))):
    print(f'  Predicted: {y_pred_labels[i]:12s} | Actual: {y_segment_test.iloc[i]}')

✅ Pipeline verification successful!
Accuracy on test set: 0.2590

Sample predictions (first 10):
  Predicted: New          | Actual: VIP
  Predicted: Returning    | Actual: VIP
  Predicted: New          | Actual: Returning
  Predicted: Returning    | Actual: Returning
  Predicted: New          | Actual: Returning
  Predicted: Returning    | Actual: Returning
  Predicted: New          | Actual: Returning
  Predicted: Returning    | Actual: Returning
  Predicted: New          | Actual: VIP
  Predicted: Returning    | Actual: VIP


## Step 4: Save the Model Artifact

We serialize the final pipeline using **joblib** as `model.joblib`. This format is required for compatibility with Vertex AI's pre-built Scikit-learn container images.

In [7]:
# Save the unified pipeline as model.joblib
model_path = os.path.join(ARTIFACTS_DIR, 'model.joblib')
joblib.dump(final_pipeline, model_path)

# Verify the saved model
loaded_pipeline = joblib.load(model_path)
y_pred_verify = loaded_pipeline.predict(X_test_raw[:5])

print(f'\u2705 Model saved as: {os.path.abspath(model_path)}')
print(f'   File size: {os.path.getsize(model_path) / 1024:.1f} KB')
print(f'\nVerification predictions: {le_segment.inverse_transform(y_pred_verify)}')

# Also save the label encoder for decoding predictions
joblib.dump(le_segment, os.path.join(ARTIFACTS_DIR, 'le_segment_deployment.pkl'))
print('\u2705 Label encoder saved for prediction decoding.')

✅ Model saved as: D:\MLFINAL\malikgit\TeamAsyncMLProject\artifacts\model.joblib
   File size: 5.1 KB

Verification predictions: ['New' 'Returning' 'New' 'Returning' 'New']
✅ Label encoder saved for prediction decoding.


In [8]:
# Verify requirements.txt exists
req_path = os.path.join(ARTIFACTS_DIR, 'requirements.txt')
print('=== requirements.txt ===')
with open(req_path, 'r') as f:
    print(f.read())

print('\n\u2705 Deployment artifacts ready:')
print(f'  1. {model_path}')
print(f'  2. {req_path}')

=== requirements.txt ===
scikit-learn==1.8.0
pandas==2.3.1
numpy==2.3.1
joblib==1.5.3
imbalanced-learn==0.14.1


✅ Deployment artifacts ready:
  1. ..\artifacts\model.joblib
  2. ..\artifacts\requirements.txt


## Step 5: Upload Artifacts to Google Cloud Storage

The following code uploads `model.joblib` and `requirements.txt` to a Google Cloud Storage (GCS) bucket. This bucket will be referenced when importing the model into Vertex AI.

### Prerequisites
- Google Cloud SDK installed and authenticated (`gcloud auth login`)
- A GCS bucket created (e.g., `gs://auracart-ml-models/`)
- Billing enabled on the GCP project

> **Note:** The code cell below requires GCP credentials. Run it in your GCP-enabled environment.

In [9]:
# === Google Cloud Storage Upload ===
# Uncomment and configure the following when deploying to GCP

# from google.cloud import storage

# # Configuration
# PROJECT_ID = 'your-gcp-project-id'  # Replace with your GCP project ID
# BUCKET_NAME = 'auracart-ml-models'   # Replace with your bucket name
# MODEL_DIR = 'customer_segment_model/v1'

# # Initialize GCS client
# client = storage.Client(project=PROJECT_ID)
# bucket = client.bucket(BUCKET_NAME)

# # Upload model.joblib
# blob_model = bucket.blob(f'{MODEL_DIR}/model.joblib')
# blob_model.upload_from_filename(os.path.join(ARTIFACTS_DIR, 'model.joblib'))
# print(f'Uploaded: gs://{BUCKET_NAME}/{MODEL_DIR}/model.joblib')

# # Upload requirements.txt
# blob_req = bucket.blob(f'{MODEL_DIR}/requirements.txt')
# blob_req.upload_from_filename(os.path.join(ARTIFACTS_DIR, 'requirements.txt'))
# print(f'Uploaded: gs://{BUCKET_NAME}/{MODEL_DIR}/requirements.txt')

# print('\n\u2705 All artifacts uploaded to GCS!')

print('\u26a0\ufe0f GCS upload code is commented out.')
print('Uncomment and configure with your GCP credentials to upload.')
print('\nRequired files in GCS bucket:')
print('  gs://<BUCKET_NAME>/customer_segment_model/v1/model.joblib')
print('  gs://<BUCKET_NAME>/customer_segment_model/v1/requirements.txt')

⚠️ GCS upload code is commented out.
Uncomment and configure with your GCP credentials to upload.

Required files in GCS bucket:
  gs://<BUCKET_NAME>/customer_segment_model/v1/model.joblib
  gs://<BUCKET_NAME>/customer_segment_model/v1/requirements.txt


## Step 6: Deploy to Vertex AI

### Deployment Steps:

1. **Import Model** into Vertex AI Model Registry from GCS
2. **Select Container** — Scikit-learn pre-built prediction container
3. **Deploy to Endpoint** — Vertex AI manages the infrastructure
4. **Test the Endpoint** — Send a JSON prediction request

### Pre-built Container
Vertex AI provides a Scikit-learn pre-built container that:
- Automatically loads `model.joblib` from GCS
- Provides an HTTP server for serving predictions
- Handles request/response serialization

> **Note:** The code cells below require GCP credentials and Vertex AI API enabled.

In [10]:
# === Vertex AI Deployment ===
# Uncomment and configure the following when deploying to Vertex AI

# from google.cloud import aiplatform

# # Initialize Vertex AI
# aiplatform.init(
#     project=PROJECT_ID,
#     location='us-central1'  # or your preferred region
# )

# # Step 1: Import the model
# model = aiplatform.Model.upload(
#     display_name='auracart-customer-segment-classifier',
#     artifact_uri=f'gs://{BUCKET_NAME}/{MODEL_DIR}',
#     serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest',
# )
# print(f'Model uploaded: {model.display_name}')
# print(f'Model resource name: {model.resource_name}')

# # Step 2: Deploy to an endpoint
# endpoint = model.deploy(
#     deployed_model_display_name='auracart-segment-v1',
#     machine_type='n1-standard-2',
#     min_replica_count=1,
#     max_replica_count=1,
# )
# print(f'\nEndpoint created: {endpoint.display_name}')
# print(f'Endpoint resource name: {endpoint.resource_name}')

print('\u26a0\ufe0f Vertex AI deployment code is commented out.')
print('Uncomment and configure with your GCP credentials to deploy.')
print('\nDeployment checklist:')
print('  1. GCP project with billing enabled')
print('  2. Vertex AI API enabled')
print('  3. Model artifacts uploaded to GCS bucket')
print('  4. Scikit-learn pre-built container selected')

⚠️ Vertex AI deployment code is commented out.
Uncomment and configure with your GCP credentials to deploy.

Deployment checklist:
  1. GCP project with billing enabled
  2. Vertex AI API enabled
  3. Model artifacts uploaded to GCS bucket
  4. Scikit-learn pre-built container selected


## Step 7: Test the Live Endpoint

We send a prediction request with a JSON payload representing a new e-commerce transaction. The endpoint should return the predicted customer segment.

In [11]:
# === Test Endpoint ===
# Uncomment when endpoint is deployed

# # Sample e-commerce transaction for prediction
# test_instance = {
#     'category': 'Electronics',
#     'quantity': 2,
#     'payment_method': 'Credit Card',
#     'device_type': 'Mobile',
#     'channel': 'Organic',
#     'order_month': 6,
#     'order_day_of_week': 3,
#     'order_hour': 14,
#     'shipping_delay_days': 2.5
# }

# # Send prediction request
# prediction = endpoint.predict(instances=[list(test_instance.values())])
# predicted_class = le_segment.inverse_transform(prediction.predictions)

# print('=== Prediction Request ===')
# print(json.dumps(test_instance, indent=2))
# print(f'\n=== Prediction Response ===')
# print(f'Predicted Customer Segment: {predicted_class[0]}')

print('--- Local Test (simulating endpoint) ---')

--- Local Test (simulating endpoint) ---


In [12]:
# Local simulation of the endpoint prediction
# This demonstrates the exact behavior the Vertex AI endpoint would produce

loaded_model = joblib.load(os.path.join(ARTIFACTS_DIR, 'model.joblib'))

# Create a sample transaction (matching the features expected by the pipeline)
sample_data = pd.DataFrame([{
    'category': 'Electronics',
    'quantity': 2,
    'payment_method': 'Credit Card',
    'device_type': 'Mobile',
    'channel': 'Organic',
    'order_month': 6,
    'order_day_of_week': 3,
    'order_hour': 14,
    'shipping_delay_days': 2.5
}])

print('=== Sample Transaction ===')
print(sample_data.to_string(index=False))

# Predict
prediction = loaded_model.predict(sample_data)
predicted_segment = le_segment.inverse_transform(prediction)

print(f'\n=== Prediction ===')
print(f'Predicted Customer Segment: {predicted_segment[0]}')

# Test with multiple samples
print('\n=== Batch Prediction Test ===')
test_samples = pd.DataFrame([
    {'category': 'Electronics', 'quantity': 1, 'payment_method': 'Credit Card',
     'device_type': 'Mobile', 'channel': 'Organic', 'order_month': 3,
     'order_day_of_week': 1, 'order_hour': 10, 'shipping_delay_days': 1.0},
    {'category': 'Clothing', 'quantity': 5, 'payment_method': 'PayPal',
     'device_type': 'Desktop', 'channel': 'Email', 'order_month': 12,
     'order_day_of_week': 5, 'order_hour': 20, 'shipping_delay_days': 5.0},
    {'category': 'Beauty', 'quantity': 3, 'payment_method': 'Apple Pay',
     'device_type': 'Tablet', 'channel': 'Social', 'order_month': 7,
     'order_day_of_week': 0, 'order_hour': 8, 'shipping_delay_days': 0.5},
])

batch_predictions = loaded_model.predict(test_samples)
batch_labels = le_segment.inverse_transform(batch_predictions)

for i in range(len(test_samples)):
    print(f'  Transaction {i+1}: {batch_labels[i]}')

print('\n\u2705 Pipeline accepts raw input and produces predictions successfully!')

=== Sample Transaction ===
   category  quantity payment_method device_type channel  order_month  order_day_of_week  order_hour  shipping_delay_days
Electronics         2    Credit Card      Mobile Organic            6                  3          14                  2.5

=== Prediction ===
Predicted Customer Segment: Returning

=== Batch Prediction Test ===
  Transaction 1: Returning
  Transaction 2: New
  Transaction 3: New

✅ Pipeline accepts raw input and produces predictions successfully!


## Summary

In this notebook, we have:

1. **Reviewed MLflow experiments** and selected the champion model for customer segment prediction
2. **Created a unified Scikit-learn Pipeline** combining the preprocessing pipeline and trained classifier
3. **Verified the pipeline** works end-to-end with raw input data
4. **Serialized the model** as `model.joblib` using joblib
5. **Documented the GCS upload** and Vertex AI deployment process
6. **Tested the prediction pipeline** with sample e-commerce transactions

### Deployment Architecture
```
Raw Transaction Data
        ↓
  [Vertex AI Endpoint]
        ↓
  [model.joblib]
    ├── Preprocessor (ColumnTransformer)
    │   ├── StandardScaler (numerical features)
    │   └── OneHotEncoder (categorical features)
    └── LogisticRegression (multinomial/softmax)
        ↓
  Predicted Customer Segment
  (New / Returning / VIP)
```

### Next Steps for Production:
1. Uncomment and run the GCS upload cells with your GCP credentials
2. Deploy to Vertex AI and capture screenshots of the endpoint
3. Send test predictions and capture the response screenshots
4. Include all screenshots in the final technical report